# Merge Partitioned BioT5 Collection Parts On Kaggle

This notebook resolves the three partitioned BioT5 collection stage artifacts, validates that they belong to the same collection run, merges their staged outputs, and exports one final merged artifact bundle for downstream training notebooks.


In [ ]:
from pathlib import Path
import sys

REPO_URL = "https://github.com/mruniverse8/Thesis.git"
REPO_BRANCH = "gflownet"
REPO_DIR = Path("/kaggle/working/Thesis")
STAGE_NAME = "merge_biot5_collection_parts"

%cd /kaggle/working
!if [ -d "{REPO_DIR / '.git'}" ]; then echo "Reusing {REPO_DIR}"; elif [ -d "{REPO_DIR}" ]; then echo "Existing non-git directory at {REPO_DIR}; delete it and rerun the notebook." && false; else git clone --depth 1 "{REPO_URL}" "{REPO_DIR}"; fi
!git -C "{REPO_DIR}" fetch --depth 1 origin "{REPO_BRANCH}" && git -C "{REPO_DIR}" checkout -B "{REPO_BRANCH}" FETCH_HEAD

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from data_collection import merge_biot5_collection_parts
from thesis_kaggle_support import (
    copy_stage_artifact_to_local,
    ensure_paths_exist,
    ensure_runtime_dependencies,
    export_stage_artifacts,
    json_dumps,
    read_json,
    report_runtime,
)


In [ ]:
NUM_PARTS = 3
PART_STAGE_NAMES = [f"collect_chebi_biot5_part_{part_index}" for part_index in range(1, NUM_PARTS + 1)]
PART_LOCAL_ROOT = REPO_DIR / "kaggle" / "generated_data" / "collection_part_artifacts"
CHEBI_PROCESSED_DIR = REPO_DIR / "data" / "chebi20" / "processed"
MERGED_COLLECTION_OUTPUT_DIR = REPO_DIR / "data_collection" / "outputs" / "kaggle_chebi20_biot5_train_merged"
MERGED_DERIVED_TRAIN_FILE = REPO_DIR / "data" / "post_training" / "processed" / "train_multimol.jsonl"

ensure_runtime_dependencies(REPO_DIR)
runtime_report = report_runtime(require_gpu=True)
print(json_dumps({
    "runtime": runtime_report,
    "part_stage_names": PART_STAGE_NAMES,
    "merged_collection_output_dir": str(MERGED_COLLECTION_OUTPUT_DIR),
    "merged_derived_train_file": str(MERGED_DERIVED_TRAIN_FILE),
}))


In [ ]:
part_runs = []
for part_index, stage_name in enumerate(PART_STAGE_NAMES, start=1):
    local_collection_dir = PART_LOCAL_ROOT / f"part_{part_index}" / "collection_outputs"
    local_derived_file = PART_LOCAL_ROOT / f"part_{part_index}" / "post_training_processed" / "train_multimol.jsonl"
    copied_collection_dir = None
    copied_derived_file = None

    if not local_collection_dir.exists():
        copied_collection_dir = copy_stage_artifact_to_local(
            stage_name=stage_name,
            artifact_relpath="collection_outputs",
            local_path=local_collection_dir,
        )
    if not local_derived_file.exists():
        copied_derived_file = copy_stage_artifact_to_local(
            stage_name=stage_name,
            artifact_relpath="post_training_processed/train_multimol.jsonl",
            local_path=local_derived_file,
        )

    if not local_collection_dir.exists() or not local_derived_file.exists():
        raise FileNotFoundError(
            f"Could not resolve collection artifacts for {stage_name}. Attach all part datasets or run the part notebooks first."
        )

    part_runs.append({
        "part_index": part_index,
        "stage_name": stage_name,
        "copied_collection_dir": None if copied_collection_dir is None else str(copied_collection_dir),
        "copied_derived_file": None if copied_derived_file is None else str(copied_derived_file),
        "collection_output_dir": str(local_collection_dir),
        "derived_train_file": str(local_derived_file),
    })

copied_processed_dir = None
if not CHEBI_PROCESSED_DIR.exists():
    copied_processed_dir = copy_stage_artifact_to_local(
        stage_name=PART_STAGE_NAMES[0],
        artifact_relpath="chebi20_processed",
        local_path=CHEBI_PROCESSED_DIR,
    )
if not CHEBI_PROCESSED_DIR.exists():
    raise FileNotFoundError(
        "Could not resolve chebi20_processed from the first part artifact."
    )

print(json_dumps({
    "copied_processed_dir": None if copied_processed_dir is None else str(copied_processed_dir),
    "part_runs": part_runs,
}))


In [ ]:
merged_summary = merge_biot5_collection_parts(
    part_staging_dirs=[item["collection_output_dir"] for item in part_runs],
    part_derived_train_files=[item["derived_train_file"] for item in part_runs],
    output_dir=MERGED_COLLECTION_OUTPUT_DIR,
    merged_derived_train_file=MERGED_DERIVED_TRAIN_FILE,
)
print(json_dumps({"merged_summary": merged_summary}))


In [ ]:
required_outputs = ensure_paths_exist({
    "chebi_processed_dir": CHEBI_PROCESSED_DIR,
    "collection_output_dir": MERGED_COLLECTION_OUTPUT_DIR,
    "collection_summary": MERGED_COLLECTION_OUTPUT_DIR / "summary.json",
    "derived_train_file": MERGED_DERIVED_TRAIN_FILE,
})
artifact_dir, manifest = export_stage_artifacts(
    stage_name=STAGE_NAME,
    artifact_map={
        "chebi20_processed": CHEBI_PROCESSED_DIR,
        "collection_outputs": MERGED_COLLECTION_OUTPUT_DIR,
        "post_training_processed/train_multimol.jsonl": MERGED_DERIVED_TRAIN_FILE,
    },
    metadata={
        "required_outputs": required_outputs,
        "part_stage_names": PART_STAGE_NAMES,
        "summary": read_json(MERGED_COLLECTION_OUTPUT_DIR / "summary.json"),
    },
)
print(json_dumps({
    "artifact_dir": str(artifact_dir),
    "manifest": manifest,
}))
